# Smart Water Distribution Analytics System
## CDAC Mumbai - AI Compute Platform
Complete PySpark Solution (Q1–Q7)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName("Smart Water Distribution Analytics").getOrCreate()
sc = spark.sparkContext
print(sc)


<SparkContext master=local[*] appName=Smart Water Distribution Analytics>


## Q1 - Load Dataset

In [2]:
df = spark.read.csv(
    "/content/DataSetAICompuetRelab.csv",
    header=True,
    inferSchema=True
)

df.printSchema()
df.show(10)
print("Total Records:", df.count())
df.describe().show()


root
 |-- Record_ID: integer (nullable = true)
 |-- Supply_Date: date (nullable = true)
 |-- Zone: string (nullable = true)
 |-- Distribution_Station: string (nullable = true)
 |-- Water_Source: string (nullable = true)
 |-- Supply_Volume_Liters: double (nullable = true)
 |-- Consumption_Liters: double (nullable = true)
 |-- Water_Loss_Liters: double (nullable = true)
 |-- Pressure_PSI: double (nullable = true)
 |-- Pipeline_ID: string (nullable = true)
 |-- Leakage_Events: integer (nullable = true)
 |-- Water_Quality: string (nullable = true)
 |-- PH_Value: double (nullable = true)
 |-- Turbidity_NTU: double (nullable = true)
 |-- Consumer_Count: integer (nullable = true)
 |-- Maintenance_Cost: double (nullable = true)
 |-- System_Status: string (nullable = true)

+---------+-----------+-------+--------------------+--------------+--------------------+------------------+-----------------+------------+-----------+--------------+-------------+--------+-------------+--------------+-------

## Q2 - RDD Programming

In [3]:
rdd = df.rdd

zone_rdd = rdd.map(lambda x: x.Zone)
high_loss = rdd.filter(lambda x: x.Water_Loss_Liters > 10000)
flat = rdd.flatMap(lambda x: x.Zone.split())

print("RDD Count:", rdd.count())
print(zone_rdd.collect()[:10])

total_supply = rdd.map(lambda x: x.Supply_Volume_Liters).reduce(lambda a,b:a+b)
print("Total Supply:", total_supply)

zone_supply = rdd.map(lambda x:(x.Zone,x.Supply_Volume_Liters)).reduceByKey(lambda a,b:a+b)
print(zone_supply.collect())


RDD Count: 10000
['Central', 'North', 'North', 'North', 'Central', 'North', 'North', 'South', 'South', 'West']
Total Supply: 1057094235.6399984
[('Central', 210179145.88000005), ('North', 214628262.21000025), ('South', 206422486.65000013), ('West', 214516964.20000035), ('East', 211347376.70000002)]


## Q3 - DataFrame Operations

In [4]:
df.select(
    "Zone",
    "Distribution_Station",
    "Supply_Volume_Liters",
    "Consumption_Liters"
).show()

df.filter(
    (col("Water_Loss_Liters")>1000) &
    (col("Leakage_Events")>2)
).show()

df.orderBy(col("Water_Loss_Liters").desc()).show()

df.groupBy("Zone").count().show()

df.groupBy("Zone").agg(
    sum("Supply_Volume_Liters").alias("Total Supply"),
    sum("Consumption_Liters").alias("Total Consumption"),
    sum("Water_Loss_Liters").alias("Total Loss"),
    avg("Pressure_PSI").alias("Average Pressure"),
    avg("Maintenance_Cost").alias("Average Cost")
).show()


+-------+--------------------+--------------------+------------------+
|   Zone|Distribution_Station|Supply_Volume_Liters|Consumption_Liters|
+-------+--------------------+--------------------+------------------+
|Central|                DS27|            72854.04|           63503.1|
|  North|                DS11|            10989.33|           9112.23|
|  North|                DS11|           163173.87|         150019.45|
|  North|                DS12|           130479.58|         113023.42|
|Central|                DS10|            87431.83|          66273.91|
|  North|                 DS1|           152778.64|         120693.18|
|  North|                DS19|           157997.93|         114399.66|
|  South|                DS14|            69118.79|          59602.42|
|  South|                DS10|           139833.28|         103052.95|
|   West|                DS25|           157212.05|         139306.36|
|  North|                 DS7|            115749.7|         113319.37|
|   Ea

## Q4 - Spark SQL

In [5]:
df.createOrReplaceTempView("water")

spark.sql("""
SELECT Zone,SUM(Consumption_Liters) AS TotalConsumption
FROM water
GROUP BY Zone
ORDER BY TotalConsumption DESC
LIMIT 5
""").show()

spark.sql("""
SELECT Zone,
SUM(Water_Loss_Liters) TotalLoss,
AVG(Water_Loss_Liters) AvgLoss,
MAX(Water_Loss_Liters) MaxLoss
FROM water
GROUP BY Zone
""").show()

spark.sql("""
SELECT Distribution_Station,
ROUND(SUM(Consumption_Liters)/SUM(Supply_Volume_Liters)*100,2) AS Efficiency
FROM water
GROUP BY Distribution_Station
ORDER BY Efficiency DESC
""").show()

spark.sql("""
SELECT month(Supply_Date) Month,
SUM(Supply_Volume_Liters) Supply
FROM water
GROUP BY month(Supply_Date)
ORDER BY Month
""").show()

spark.sql("""
SELECT month(Supply_Date) Month,
SUM(Consumption_Liters) Consumption
FROM water
GROUP BY month(Supply_Date)
ORDER BY Month
""").show()

spark.sql("""
SELECT Zone,
SUM(Leakage_Events) Leakages
FROM water
GROUP BY Zone
ORDER BY Leakages DESC
""").show()

spark.sql("""
SELECT Water_Source,
AVG(PH_Value) AvgPH,
AVG(Turbidity_NTU) AvgTurbidity
FROM water
GROUP BY Water_Source
""").show()


+-------+--------------------+
|   Zone|    TotalConsumption|
+-------+--------------------+
|  North|1.7521778835000008E8|
|   West|1.7504741414000028E8|
|   East|      1.7249534199E8|
|Central| 1.712170387399998E8|
|  South| 1.668846068800003E8|
+-------+--------------------+

+-------+-------------------+------------------+--------+
|   Zone|          TotalLoss|           AvgLoss| MaxLoss|
+-------+-------------------+------------------+--------+
|  South|3.953787977000007E7|20019.179630379782| 68634.1|
|Central|3.896210713999991E7|19648.062097831524|68037.43|
|   East|3.885203470999998E7|19435.735222611296|68424.59|
|   West|3.946955005999999E7|19357.307533104457|68575.26|
|  North|3.941047385999994E7| 19665.90511976045|68736.19|
+-------+-------------------+------------------+--------+

+--------------------+----------+
|Distribution_Station|Efficiency|
+--------------------+----------+
|                DS18|     82.75|
|                DS21|     82.72|
|                DS20|     

## Q5 - ETL Pipeline

In [6]:
etl = spark.read.csv(
    "/content/DataSetAICompuetRelab.csv",
    header=True,
    inferSchema=True
)

etl = etl.dropDuplicates()
etl = etl.na.fill(0)
etl = etl.withColumn("Supply_Date", to_date("Supply_Date"))

etl.write.mode("overwrite").parquet("Water_Data_Parquet")

print("ETL Completed")


ETL Completed


### ETL Workflow

CSV → Extract → Remove Duplicates → Handle Missing Values → Convert Data Types → Save as Parquet


## Q6 - Spark MLlib

In [7]:
assembler = VectorAssembler(
    inputCols=[
        "Supply_Volume_Liters",
        "Pressure_PSI",
        "Consumer_Count",
        "Leakage_Events"
    ],
    outputCol="features"
)

data = assembler.transform(df)

train,test = data.randomSplit([0.8,0.2],seed=1)

lr = LinearRegression(
    featuresCol="features",
    labelCol="Consumption_Liters"
)

model = lr.fit(train)

prediction = model.transform(test)
prediction.select("Consumption_Liters","prediction").show()

evaluator = RegressionEvaluator(
    labelCol="Consumption_Liters",
    predictionCol="prediction",
    metricName="rmse"
)

print("RMSE:", evaluator.evaluate(prediction))


+------------------+------------------+
|Consumption_Liters|        prediction|
+------------------+------------------+
|          66273.91| 71128.15286622489|
|         106283.75|127745.11772226704|
|          93168.35|108768.43633729915|
|         175371.86|149331.63164576836|
|          16462.22|19232.739971262643|
|         106530.12|111870.68326845787|
|         188283.33| 161839.7729420883|
|          65329.58| 71257.28609832606|
|          66531.75|  74738.3742582708|
|         146849.94|139473.83759298868|
|         168148.61| 144228.7332613534|
|          57182.75|  60298.5978756341|
|          42816.11| 44213.09297497946|
|          69823.94| 67659.62031550426|
|         106113.23|131311.96880805967|
|          30749.32|30540.019295196653|
|         141618.22| 161372.5949868313|
|           70434.5| 81753.22989973066|
|          92899.85|115756.05945664768|
|         159176.03| 134507.5708368713|
+------------------+------------------+
only showing top 20 rows
RMSE: 11436.355

## Q7 - GitHub and CI/CD

In [8]:
# Git Commands

# git init
# git add .
# git commit -m "Smart Water Analytics"
# git branch -M main
# git remote add origin https://github.com/username/Smart-Water-Analytics.git
# git push -u origin main


### GitHub Actions (spark.yml)

```yaml
name: Spark CI

on:
  push:
    branches: [main]

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install pyspark pandas
      - run: python main.py
```
